# Single-cell RNA-seq analysis: human skin wound healing

**SupBioTech M2 practical** - scRNA-seq only (GEO: GSE241132).

Data from *Spatiotemporal single-cell roadmap of human skin wound healing*. We analyze the donor **PWH27**, sampled at four timepoints after wounding (D0 = intact skin, D1, D7, D30).

Outline: load -> QC and filtering -> doublets -> normalization / PCA / UMAP -> (demo of) batch correction -> clustering resolution with RESOLUTE -> marker genes -> cell type annotation -> validation against the published annotation -> cell states.

All the intermediate results are saved in `data/processed_data/SupBioTech`.

In [ ]:
import numpy as np
import pandas as pd

from matplotlib import pyplot as plt

import scanpy as sc
import anndata as ad
import seaborn as sns
import harmonypy

sns.set_style("whitegrid")

BASE = '/Users/sergio/Spatiotemporal_roadmap_of_human_skin_wound_healing'
SC_DATA = f'{BASE}/data/GSE241132_RAW'
OUT = f'{BASE}/data/processed_data/SupBioTech'   # all results of this notebook

## Single-cell data analysis

### Strategy

Each row of a scRNA-seq matrix is a **single cell**: a (relatively) pure transcriptional profile, belonging to at most one cell type. With tens of thousands of cells per donor we have enough statistical power to support fine-grained, well-separated clusters. For this reason, we invest some care in choosing the clustering resolution (with RESOLUTE, rather than guessing), and in annotating the clusters (data-driven markers + the marker genes reported in the paper).

In [ ]:
# load the single-cell data for the donor PWH27 at different timepoints
donor = "PWH27"
timepoints = ["D0", "D1", "D7", "D30"]
sc_data_folder = SC_DATA

adatas_sc = {}
for tp in timepoints:
    a = sc.read_10x_mtx(f"{sc_data_folder}/{donor}{tp}") 
    a.obs["donor"] = donor
    a.obs["sample"] = tp 
    a.obs_names = [f"{bc}-{tp}" for bc in a.obs_names]
    adatas_sc[tp] = a

adata_sc = ad.concat(list(adatas_sc.values()), join="inner")

In [ ]:
adata_sc

In [ ]:
# compute QC metrics first (same as before)

adata_sc.var["mt"] = adata_sc.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata_sc, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)

METRICS = ["total_counts", "n_genes_by_counts", "pct_counts_mt"]
TITLES = {
    "total_counts": "Total UMIs per cell",
    "n_genes_by_counts": "Detected genes per cell",
    "pct_counts_mt": "% mitochondrial counts per cell",
}
SC_SAMPLES = ["D0", "D1", "D7", "D30"]  # matches your actual "sample" column values

In [ ]:
# adata_sc.obs

In [ ]:
# Classic violin plots, one panel per sample ---
for s in SC_SAMPLES:
    a = adata_sc[adata_sc.obs["sample"] == s]
    fig, axes = plt.subplots(1, len(METRICS), figsize=(12, 4))
    for ax, metric in zip(axes, METRICS):
        sns.violinplot(y=a.obs[metric], ax=ax, inner="box", cut=0)
        sns.stripplot(y=a.obs[metric], ax=ax, color="black", size=1.5, alpha=0.3)
        ax.set_title(TITLES[metric])
        ax.set_ylabel("")
    fig.suptitle(f"QC per sample: {s} (n={a.n_obs} cells)")
    fig.tight_layout()
    plt.show()

In [ ]:
# Combined view across samples (same as the ST "QC combined" cell)
fig, axes = plt.subplots(1, len(METRICS), figsize=(14, 4))
for ax, metric in zip(axes, METRICS):
    sns.violinplot(
        data=adata_sc.obs, x="sample", y=metric, order=SC_SAMPLES,
        ax=ax, inner="box", cut=0,
    )
    ax.set_title(TITLES[metric])
    ax.set_xlabel("")
    ax.set_ylabel("")
    ax.tick_params(axis="x", rotation=30)
fig.suptitle("QC combined across samples (donor PWH27)")
fig.tight_layout()
plt.show()

In [ ]:

# Then apply your filtering thresholds below, and the custom filters for this specific dataset (MALAT1, mitochondrial genes, hemoglobin genes)

sc.pp.filter_cells(adata_sc, min_genes=500) # same filter than in the paper
sc.pp.filter_cells(adata_sc, min_counts=1000) # same filter than in the paper
adata_sc = adata_sc[adata_sc.obs["pct_counts_mt"] < 20].copy() # same filter than in the paper

In [ ]:
# Doublet removal with scrublet 
sc.pp.scrublet(adata_sc, batch_key="sample")
print(adata_sc.obs["predicted_doublet"].value_counts())

In [ ]:
n_before = adata_sc.n_obs
adata_sc = adata_sc[~adata_sc.obs["predicted_doublet"]].copy()
print(f"Doublet removal (your own Scrublet run): {n_before} -> {adata_sc.n_obs} cells")

In [ ]:
# Gene level filtering: remove MALAT1, mitochondrial genes, and hemoglobin genes
hb_genes = ["HBA1","HBA2","HBB","HBD","HBE1","HBG1","HBG2","HBM","HBQ1","HBZ"]
mt_genes = adata_sc.var_names[adata_sc.var_names.str.startswith("MT-")].tolist()
remove = set(["MALAT1"] + mt_genes + hb_genes) & set(adata_sc.var_names)
adata_sc = adata_sc[:, ~adata_sc.var_names.isin(remove)].copy()

sc.pp.filter_genes(adata_sc, min_cells=10)

In [ ]:
# Normalization, HVG, PCA, uncorrected UMAP:
adata_sc.layers["counts"] = adata_sc.X.copy()
sc.pp.normalize_total(adata_sc, target_sum=1e4)
sc.pp.log1p(adata_sc)
sc.pp.highly_variable_genes(adata_sc, n_top_genes=2000, batch_key="sample")
sc.tl.pca(adata_sc, n_comps=30, use_highly_variable=True)
sc.pp.neighbors(adata_sc, n_pcs=30)
sc.tl.umap(adata_sc)

sc.pl.umap(adata_sc, color="sample", title="uncorrected (single donor)")

In [ ]:
# save the anndatas at this point of the analysis
adata_sc.write_h5ad(
    f"{OUT}/sc_data_PWH27_processed.h5ad"
)

In [ ]:
# Batch correction demo
ho_sc = harmonypy.run_harmony(adata_sc.obsm["X_pca"], adata_sc.obs, ["sample"])
adata_sc.obsm["X_pca_harmony"] = ho_sc.Z_corr
sc.pp.neighbors(adata_sc, use_rep="X_pca_harmony", key_added="harmony")
sc.tl.umap(adata_sc, neighbors_key="harmony")
sc.pl.umap(adata_sc, neighbors_key="harmony", color="sample", title="harmony (single donor)")

In [ ]:
## Calculate again the UMAP on the uncorrected graph 

sc.tl.umap(adata_sc)  
sc.pl.umap(adata_sc, color="sample", title="uncorrected (recomputed)")

In [ ]:
import resolute as rs

results_calinski = rs.run_resolute(
    adata=adata_sc,
    score_value="calinski",
    use_rep="X_pca",  # uncorrected data, as decided
    min_res=0.1, max_res=2.0, step=0.1,
    clustering_algorithm="leiden",
)
best_resolution = results_calinski["best_score_res"]

In [ ]:
results_bic = rs.run_resolute(
    adata=adata_sc,
    score_value="bic",
    use_rep="X_pca",
    compute_stability=False,       # do not recompute bootstrap stability for UMAP
    min_res=0.1, max_res=2.0, step=0.1,
    clustering_algorithm="leiden",
)

## Choosing the clustering resolution

We used RESOLUTE to test three independent criteria for selecting the leiden clustering resolution: BIC, Calinski-Harabasz, and bootstrap stability.

Two out of three criteria (Calinski-Harabasz and stability) agree on **resolution 0.2** as the optimal choice. BIC instead keeps decreasing across the whole tested range without showing a clear turning point, which suggests it did not find a genuine optimum within these settings.

As a first pass, we proceed with **resolution 0.2**, since it is supported by two independent and conceptually different criteria. We will treat this as a working choice: the real test is whether the resulting clusters map onto coherent, biologically interpretable cell types once we look at marker genes (`rank_genes_groups`), and not the resolution value itself.

In [ ]:
# Final leiden clustering at the resolution selected by RESOLUTE
# (agreement between Calinski-Harabasz and bootstrap stability)
sc.tl.leiden(
    adata_sc,
    neighbors_key="neighbors",
    resolution=0.2,
    key_added="leiden",
    flavor="igraph",
    directed=False,
)
print(adata_sc.obs["leiden"].value_counts())

sc.pl.embedding(
    adata_sc, basis="X_umap", color="leiden",
    title="Leiden clusters (resolution=0.2, uncorrected)",
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15, 5))
sc.pl.embedding(adata_sc, basis="X_umap", color="sample", ax=axes[0], show=False, title="colored by sample")
sc.pl.embedding(adata_sc, basis="X_umap", color="leiden", ax=axes[1], show=False, title="Leiden clusters (resolution=0.2)")

fig.tight_layout()
plt.show()

In [ ]:
# 1) Differential expression between clusters (one-vs-rest, classic approach)
sc.tl.rank_genes_groups(
    adata_sc, groupby="leiden", method="wilcoxon", key_added="rank_genes_leiden"
)

# 2) Dotplot of the data-driven top marker genes per cluster
#    (scanpy pulls the top N genes directly from the rank_genes_groups result)
sc.tl.dendrogram(adata_sc, groupby="leiden", use_rep="X_pca")
sc.pl.rank_genes_groups_dotplot(
    adata_sc, groupby="leiden", key="rank_genes_leiden",
    n_genes=5, standard_scale="var",
)

In [ ]:
marker_table = pd.read_csv(
    f"{BASE}/data/paper_marker_genes.csv"  # adjust path as needed
)

# Build a marker dict usable with sc.pl.dotplot, e.g. for the sc major cell types
sc_major_markers = {
    row["cell_type"]: row["marker_genes"].split(";")
    for _, row in marker_table.query("dataset == 'scRNA-seq' and granularity == 'major'").iterrows()
}

sc.pl.dotplot(adata_sc, sc_major_markers, groupby="leiden", standard_scale="var", dendrogram=True)

In [ ]:
# marker_table

In [ ]:
# Save checkpoint: save the anndata object with the clustering and marker results
adata_sc.write_h5ad(
    f"{OUT}/adata_sc_leiden_res0.2.h5ad"
)

### Cell type assignment - from clusters to cell types

Leiden clustering gives us a fine-grained partition of the data (here, 17 clusters) based purely on transcriptional similarity. These clusters are not yet cell types — they are just groups of cells that look alike to the algorithm.

Cell type assignment is a separate, manual step: for each cluster, we look at its marker genes (from the differential expression test) and known marker gene sets (e.g., from the literature) to decide which biological cell type it most likely represents.

It is expected that several clusters end up mapping to the **same** cell type (e.g., different activation states or conditions of the same cell population). This is a many-to-one relationship, not one-to-one,and the paper itself follows the same logic: 27 fine-grained clusters (full dataset, Louvain resolution 0.8) are grouped into only 9 major cell types based on marker genes. Our 17 clusters are at a comparable "fine" level, and we expect them to collapse into a similarly small number of major cell types once annotated.

In [ ]:
# Top 5 marker genes per cluster, readable table
top5 = (
    sc.get.rank_genes_groups_df(adata_sc, group=None, key="rank_genes_leiden")
    .sort_values("scores", ascending=False)
    .groupby("group")
    .head(5)
).sort_values(by=["group", "logfoldchanges"], ascending=[True, False]) #sort by group first, then by logfoldchanges within each group
print(top5[["group", "names", "logfoldchanges"]].to_string())

In [ ]:
paper_meta = pd.read_csv(f"{SC_DATA}/GSE241132_cell_metadata.txt", sep="\t", index_col=0)

print("Paper metadata barcode format (first 5):")
print(paper_meta.index[:5].tolist())

print("\nYour adata_sc barcode format (first 5):")
print(adata_sc.obs_names[:5].tolist())

print("\nPaper metadata columns:", paper_meta.columns.tolist())


## Cluster annotation: marker-based interpretation

For each Leiden cluster, we inspected its top differentially expressed genes (one-vs-rest Wilcoxon test) and matched them against known literature markers to assign a provisional major cell type. The table below summarizes this first-pass, marker-based annotation.

| Cluster | Top marker genes | Proposed cell type | Notes |
|---|---|---|---|
| 0 | KRT15, DST, COL17A1, MIR205HG | Keratinocytes | Basal state |
| 16 | S100A2, COL17A1, KRT5, KRT14 | Keratinocytes | Basal state |
| 11 | SPINK5, CALML5, SLURP1, SBSN | Keratinocytes | Differentiated / granular state |
| 2 | LGALS7, LGALS7B, KRT1, PERP | Keratinocytes | Suprabasal state |
| 6 | RPLP0, MT1X, KRT5, SNHG29 | Keratinocytes | Ribosomal/metallothionein-dominated signature. Possibly a lower-quality or stress-associated cluster rather than a distinct biological state |
| 15 | S100A14, MT1G, LY6D, S100A9 | Keratinocytes | S100A9 suggests a wound-activated state |
| 3 | HLA-DRB1, HLA-DRA, CD74, HLA-DPB1 | Myeloid cells | MHC-II high (dendritic-like) |
| 4 | FTL, CTSS, AIF1, CD68 | Myeloid cells | Macrophage-like |
| 10 | RPS27, B2M, RPL28, TMSB4X | Lymphoid cells (tentative) | Marker genes are largely non-specific / housekeeping. Needs targeted verification |
| 12 | TPSB2, HPGD, TPSAB1, HPGDS | Mast cells | Unambiguous (tryptase genes) |
| 8 | FTH1, CFD, SERPING1, C1S | Fibroblasts | |
| 13 | VIM, LGALS1, TPM2, CAVIN3 | Fibroblasts | Possible myofibroblast-like state |
| 14 | SPARC, COL3A1, COL1A1, LUM | Fibroblasts | |
| 7 | MLANA, DCT, TYRP1 | Melanocytes | Unambiguous |
| 1 | MCAM, TAGLN, CALD1, MYL9 | Pericytes and smooth muscle cells | |
| 5 | MMRN1, CCL21, TFF3 | Endothelial cells | Lymphatic endothelial state |
| 9 | PECAM1, SPARCL1, GNG11 | Endothelial cells | Vascular state |

No cluster was clearly identifiable as Schwann cells, likely because this is a rare population and we only have a single donor. These cells may be present as a small subpopulation absorbed into a larger cluster (e.g., cluster 7, since Schwann cells and melanocytes share a neural crest origin and the SOX10 marker) rather than forming a cluster of their own.

**Note on provenance:** the top marker genes come from our own differential expression results. The cell type labels were validated against the original publication's annotation at the major cell type level. The finer state descriptions in the notes column (e.g. basal, differentiated, wound-activated) are interpretations based on general knowledge of skin biology and were not individually validated against the paper or a specific reference, so they should be read as working hypotheses.

## Next step: validating the annotation against the published dataset

This marker-based annotation is a first pass, based on visual inspection of marker gene dot plots (useful, but subjective). Because our cells are a subset of the same GEO dataset the authors analyzed, we can cross-check our cluster labels against their own published per-cell annotation (`GSE241132_cell_metadata.txt`), by matching cell barcodes between the two.

This gives us an objective, quantitative comparison: for each of our Leiden clusters, how many cells the original authors already assigned to each cell type. We will reconcile the two barcode naming conventions, merge the published annotation into our `adata_sc.obs`, and build a cluster-by-cell-type cross-tabulation table to confirm (or correct) the marker-based assignment above before finalizing it.

In [ ]:
# Manually curated T/NK cell markers from the paper's Supplementary Table
sc.pl.dotplot(adata_sc, {"T/NK markers": ["CD3D", "CD3E", "NKG7"]}, groupby="leiden", standard_scale="var")

| 10 | RPS27, B2M, RPL28, TMSB4X (DE genes); CD3D/CD3E/NKG7 confirmed by targeted check | Lymphoid cells | Top-5 DE genes were non-specific; confirmed as T/NK lineage via targeted marker dotplot (CD3D, CD3E, NKG7 all clearly positive, unique to this cluster) |

In [ ]:
## Cell type annotation 
cluster_to_celltype = {
    "0": "Keratinocytes", "16": "Keratinocytes", "11": "Keratinocytes",
    "2": "Keratinocytes", "6": "Keratinocytes", "15": "Keratinocytes",
    "3": "Myeloid cells", "4": "Myeloid cells",
    "10": "Lymphoid cells",
    "12": "Mast cells",
    "8": "Fibroblasts", "13": "Fibroblasts", "14": "Fibroblasts",
    "7": "Melanocytes",
    "1": "Pericytes and smooth muscle cells",
    "5": "Endothelial cells", "9": "Endothelial cells",
}
adata_sc.obs["cell_type"] = adata_sc.obs["leiden"].map(cluster_to_celltype).astype("category")

sc.pl.embedding(adata_sc, basis="X_umap", color="cell_type", title="Cell type annotation")

#### Validating the annotation against the published dataset

The cluster annotation above was built by manually inspecting marker gene dot plots, that is a reasonable first pass, but ultimately a subjective judgment call. Because our cells are a subset of the same GEO dataset analyzed in the original publication, we can perform an objective, independent check: the authors already assigned a cell type to every cell they kept (`GSE241132_cell_metadata.txt`, column `newMainCellTypes`), so we can match our own cells back to theirs by barcode and compare.

The two barcode naming conventions differ (`{donor}{timepoint}_{barcode}` in the paper's metadata vs. `{barcode}-{timepoint}` in ours), so we first reconcile the format, then merge in the published cell type label for every cell we can match. Finally, we cross-tabulate our Leiden clusters against these published labels: for each cluster, this tells us exactly how its cells were classified by the original authors, giving us a quantitative measure of how well our unsupervised clustering + marker-based annotation agrees with ground truth.

In [ ]:
DONOR = "PWH27"

def to_paper_barcode(adata_barcode, donor=DONOR):
    core, tp = adata_barcode.rsplit("-", 1)  # "AAACCCAAGTATCCTG-1", "D0"
    return f"{donor}{tp}_{core}"

adata_sc.obs["paper_barcode"] = [to_paper_barcode(bc) for bc in adata_sc.obs_names]

paper_meta_pwh27 = paper_meta[paper_meta.index.str.startswith(DONOR)]
adata_sc.obs["paper_newMainCellTypes"] = adata_sc.obs["paper_barcode"].map(paper_meta_pwh27["newMainCellTypes"])

n_matched = adata_sc.obs["paper_newMainCellTypes"].notna().sum()
print(f"Matched {n_matched} / {adata_sc.n_obs} cells to the paper's published annotation")

# Cross-tabulation: your leiden clusters vs. the paper's own cell type calls
pd.crosstab(adata_sc.obs["leiden"], adata_sc.obs["paper_newMainCellTypes"])

#### Finalizing the cell type annotation

Given the high concordance with the published annotation (96.7% of matched cells fall into the cluster's expected majority type), we keep our original marker-based annotation as the final one.

In [ ]:
# Save checkpoint: save the anndata object with the cell type annotation and paper metadata mapping
adata_sc.write_h5ad(
    f"{OUT}/adata_sc_leiden_res0.2_annotated.h5ad"
)

In [ ]:
# load the anndata object with the cell type annotation and paper metadata mapping
#adata_sc = sc.read_h5ad(
#    f"{OUT}/adata_sc_leiden_res0.2_annotated.h5ad"
#)

In [ ]:
# Finer-grained label (state level) for the Cell2location reference,
# based on the marker-based interpretation of each Leiden cluster
cluster_to_state = {
    "0":  "Keratinocytes_basal_A",
    "16": "Keratinocytes_basal_B",
    "11": "Keratinocytes_differentiated",
    "2":  "Keratinocytes_suprabasal",
    "6":  "Keratinocytes_ribosomal_stress",   
    "15": "Keratinocytes_wound_activated",  
    "3":  "Myeloid_MHCII_high",
    "4":  "Macrophages",
    "10": "Lymphoid_T_NK",
    "12": "Mast_cells",
    "8":  "Fibroblasts_CFD",
    "13": "Fibroblasts_myofibroblast_like",
    "14": "Fibroblasts_ECM_high",
    "7":  "Melanocytes",
    "1":  "Pericytes_SMC",
    "5":  "Endothelial_lymphatic",
    "9":  "Endothelial_vascular",
}
adata_sc.obs["cell_state"] = adata_sc.obs["leiden"].map(cluster_to_state).astype("category")
print(adata_sc.obs["cell_state"].value_counts())

In [ ]:
# adata_sc.obs

In [ ]:
# save the adata for sc with the cell state:
adata_sc.write_h5ad(f"{OUT}/adata_sc_annotated_cell_state.h5ad")

In [ ]:
# load the adata_sc object with the cell state annotation
#adata_sc = sc.read_h5ad(f"{OUT}/adata_sc_annotated_cell_state.h5ad")